# Evaluating Urban Pedestrian Footfall Dynamics Using Exploratory Data Analysis and Seaborn Visualizations

**Domain:** Smart Cities & Mobility  |  **Project #:** 28  |  **COs:** CO4, CO5  |  **Level:** K4 (Analyze)  
**Stack:** Python · Pandas · Seaborn (+ Matplotlib, NumPy, SciPy)

### Problem statement
City planners need to know *when*, *where* and *why* pedestrian traffic rises and falls so they can schedule
cleaning, policing, transit frequency, retail hours and event management. This notebook performs a full
Exploratory Data Analysis (EDA) on hourly pedestrian sensor counts.

### Objectives
1. Load, inspect and clean hourly pedestrian count data from multiple sensor locations.
2. Engineer time features (hour, weekday, month, season, peak/off-peak).
3. Analyse **temporal** patterns (daily, weekly, seasonal), **spatial** patterns (by location) and **external drivers** (rain, temperature, events).
4. Detect anomalies / outliers.
5. Communicate findings with Seaborn visualisations and turn them into planning recommendations.

### Dataset note
Cell 3 lets you choose a data source:
- `USE_SYNTHETIC = True` → generates a realistic synthetic dataset (works instantly, no upload needed).
- `USE_SYNTHETIC = False` → upload your own CSV (e.g. the **Melbourne Pedestrian Counting System** from data.melbourne.vic.gov.au, or any sensor data) and map the column names.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 50)

print("pandas", pd.__version__, "| seaborn", sns.__version__)

## 2. Configuration
Switch `USE_SYNTHETIC` to `False` to use your own file. Your CSV needs at minimum a **timestamp**, a **location/sensor name** and a **count** column.

In [ ]:
USE_SYNTHETIC = True      # False -> upload your own CSV

# Only used when USE_SYNTHETIC = False
COL_TIME     = "Date_Time"
COL_LOCATION = "Sensor_Name"
COL_COUNT    = "Hourly_Counts"

RANDOM_SEED = 42

## 3. Load the data

In [ ]:
def generate_synthetic_footfall(seed=42):
    """Hourly pedestrian counts for 5 sensor locations over 2 years, with weather and events."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2023-01-01", "2024-12-31 23:00", freq="h")

    # --- City-wide drivers (same for every sensor) ---
    doy = idx.dayofyear.values
    temp = 24 + 7 * np.sin(2 * np.pi * (doy - 80) / 365) + rng.normal(0, 2, len(idx)) \
           + 3 * np.sin(2 * np.pi * (idx.hour.values - 9) / 24)
    # rain per day (so the whole day shares one state), more likely in monsoon-ish months
    days = idx.normalize()
    uniq_days = days.unique()
    rain_prob = np.where(uniq_days.month.isin([6, 7, 8, 9]), 0.35, 0.10)
    rain_day = pd.Series(rng.random(len(uniq_days)) < rain_prob, index=uniq_days)
    rain = days.map(rain_day).values.astype(int)

    # a handful of big events (festival, marathon, concert ...)
    event_days = pd.to_datetime(rng.choice(uniq_days, 14, replace=False))
    event = np.isin(days, event_days).astype(int)
    holidays = pd.to_datetime(["2023-01-26","2023-08-15","2023-10-02","2023-11-12","2023-12-25",
                               "2024-01-26","2024-08-15","2024-10-02","2024-10-31","2024-12-25"])
    holiday = np.isin(days, holidays).astype(int)

    h = idx.hour.values
    dow = idx.dayofweek.values
    weekend = dow >= 5

    def gauss(x, mu, s): return np.exp(-0.5 * ((x - mu) / s) ** 2)

    # --- Location archetypes: (base volume, weekday profile, weekend profile, rain sensitivity) ---
    locs = {
        "Central Market Street": dict(base=900,  wd=lambda x: 0.15 + 0.7*gauss(x,13,3.5) + 0.5*gauss(x,18,2),
                                       we=lambda x: 0.1 + 1.1*gauss(x,13,4), rain=-0.30, event=0.9),
        "Railway Station":       dict(base=1500, wd=lambda x: 0.1 + 1.3*gauss(x,8,1.3) + 1.2*gauss(x,18,1.6),
                                       we=lambda x: 0.1 + 0.5*gauss(x,12,4), rain=-0.10, event=0.5),
        "University Gate":       dict(base=700,  wd=lambda x: 0.05 + 1.0*gauss(x,9,1.5) + 0.7*gauss(x,13,1.5) + 0.9*gauss(x,17,1.5),
                                       we=lambda x: 0.05 + 0.15*gauss(x,12,4), rain=-0.25, event=0.1),
        "Riverside Promenade":   dict(base=500,  wd=lambda x: 0.05 + 0.4*gauss(x,7,1.5) + 0.7*gauss(x,18,2),
                                       we=lambda x: 0.05 + 1.2*gauss(x,10,2.5) + 1.3*gauss(x,17,2.5), rain=-0.60, event=1.4),
        "Tech Park Junction":    dict(base=800,  wd=lambda x: 0.05 + 1.2*gauss(x,9,1.2) + 0.5*gauss(x,13,1) + 1.2*gauss(x,18,1.5),
                                       we=lambda x: 0.05 + 0.1*gauss(x,12,4), rain=-0.20, event=0.05),
    }
    frames = []
    season_boost = 1 + 0.12 * np.sin(2 * np.pi * (doy - 300) / 365)       # slightly busier in cooler months
    for name, p in locs.items():
        prof = np.where(weekend, p["we"](h), p["wd"](h))
        prof = np.where(holiday == 1, p["we"](h) * 0.9, prof)
        temp_effect = 1 - 0.012 * np.clip(np.abs(temp - 24) - 4, 0, None)  # discomfort when too hot/cold
        mu = p["base"] * prof * season_boost * temp_effect
        mu = mu * (1 + p["rain"] * rain) * (1 + p["event"] * event * gauss(h, 17, 3))
        mu = np.clip(mu, 0.5, None)
        # negative-binomial-ish noise (overdispersed counts)
        counts = rng.negative_binomial(n=25, p=25 / (25 + mu)).astype(float)
        frames.append(pd.DataFrame({"Date_Time": idx, "Sensor_Name": name, "Hourly_Counts": counts,
                                    "Temperature_C": temp.round(1), "Rain": rain,
                                    "Event": event, "Public_Holiday": holiday}))
    df = pd.concat(frames, ignore_index=True)

    # --- Inject realistic data-quality problems for the cleaning step ---
    n = len(df)
    miss = rng.choice(n, int(0.012 * n), replace=False); df.loc[miss, "Hourly_Counts"] = np.nan
    out = df[df["Sensor_Name"] == "Railway Station"].sample(6, random_state=1).index
    df.loc[out, "Hourly_Counts"] *= 8                                   # sensor spikes
    outage = df[(df["Sensor_Name"] == "University Gate") &
                (df["Date_Time"].between("2023-06-10", "2023-06-13"))].index
    df.loc[outage, "Hourly_Counts"] = 0                                 # sensor outage (zeros)
    df = pd.concat([df, df.sample(40, random_state=3)], ignore_index=True)  # duplicates
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


if USE_SYNTHETIC:
    raw = generate_synthetic_footfall(RANDOM_SEED)
    print("Using SYNTHETIC dataset")
else:
    from google.colab import files
    up = files.upload()
    raw = pd.read_csv(next(iter(up)))
    raw = raw.rename(columns={COL_TIME: "Date_Time", COL_LOCATION: "Sensor_Name", COL_COUNT: "Hourly_Counts"})
    print("Using UPLOADED dataset")

raw.head()

## 4. Data inspection

In [ ]:
print("Shape:", raw.shape)
print("\nDtypes:\n", raw.dtypes)
print("\nMissing values:\n", raw.isna().sum())
print("\nDuplicate rows:", raw.duplicated().sum())
raw.describe(include="all").T

## 5. Data cleaning

In [ ]:
df = raw.copy()

# 1) types
df["Date_Time"] = pd.to_datetime(df["Date_Time"], errors="coerce")
df = df.dropna(subset=["Date_Time"])

# 2) duplicates
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["Date_Time", "Sensor_Name"], keep="first")
print(f"Removed {before - len(df)} duplicate rows")

# 3) sort
df = df.sort_values(["Sensor_Name", "Date_Time"]).reset_index(drop=True)

# 4) sensor outages: 24h+ of consecutive zeros -> treat as missing
def flag_outage(s, min_run=24):
    is_zero = (s == 0)
    run_id = (is_zero != is_zero.shift()).cumsum()
    run_len = is_zero.groupby(run_id).transform("sum")
    return is_zero & (run_len >= min_run)

df["Outage"] = df.groupby("Sensor_Name")["Hourly_Counts"].transform(flag_outage)
print("Outage hours flagged:", int(df["Outage"].sum()))
df.loc[df["Outage"], "Hourly_Counts"] = np.nan

# 5) fill short gaps by time-interpolation within each sensor, then drop what's left
print("Missing before fill:", int(df["Hourly_Counts"].isna().sum()))
df["Hourly_Counts"] = (df.set_index("Date_Time")
                         .groupby("Sensor_Name")["Hourly_Counts"]
                         .transform(lambda s: s.interpolate(method="time", limit=3))
                         .values)
df = df.dropna(subset=["Hourly_Counts"]).reset_index(drop=True)
print("Missing after fill (rows dropped):", int(df["Hourly_Counts"].isna().sum()))
print("Clean shape:", df.shape)

### 5.1 Handling extreme sensor spikes
We cap values using a **robust, location-and-hour specific** rule (median + 6 × MAD). This avoids deleting genuine peaks (e.g. rush hour) while removing impossible values.

In [ ]:
keys = [df["Sensor_Name"], df["Date_Time"].dt.hour, df["Date_Time"].dt.dayofweek >= 5]
grp = df.groupby(keys)["Hourly_Counts"]
med = grp.transform("median")
mad = grp.transform(lambda s: (s - s.median()).abs().median()).clip(lower=1)
upper = med + 8 * 1.4826 * mad        # robust bound per sensor x hour x weekday/weekend
spikes = df["Hourly_Counts"] > upper
print("Extreme spikes detected:", int(spikes.sum()))
display(df[spikes].head(10))
df.loc[spikes, "Hourly_Counts"] = upper[spikes]      # winsorise instead of dropping

## 6. Feature engineering

In [ ]:
d = df["Date_Time"]
df["Date"]       = d.dt.normalize()
df["Year"]       = d.dt.year
df["Month"]      = d.dt.month
df["Month_Name"] = d.dt.strftime("%b")
df["Day_Name"]   = d.dt.day_name()
df["DayOfWeek"]  = d.dt.dayofweek
df["Hour"]       = d.dt.hour
df["Is_Weekend"] = df["DayOfWeek"] >= 5
df["Day_Type"]   = np.where(df["Is_Weekend"], "Weekend", "Weekday")

season_map = {12:"Winter",1:"Winter",2:"Winter",3:"Summer",4:"Summer",5:"Summer",
              6:"Monsoon",7:"Monsoon",8:"Monsoon",9:"Monsoon",10:"Post-monsoon",11:"Post-monsoon"}
df["Season"] = df["Month"].map(season_map)

def part_of_day(h):
    if   5 <= h < 9:   return "Early Morning"
    elif 9 <= h < 12:  return "Morning"
    elif 12 <= h < 17: return "Afternoon"
    elif 17 <= h < 21: return "Evening"
    return "Night"
df["Part_of_Day"] = df["Hour"].map(part_of_day)

day_order  = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pod_order  = ["Early Morning","Morning","Afternoon","Evening","Night"]
df["Day_Name"]   = pd.Categorical(df["Day_Name"], day_order, ordered=True)
df["Month_Name"] = pd.Categorical(df["Month_Name"], month_order, ordered=True)
df["Part_of_Day"] = pd.Categorical(df["Part_of_Day"], pod_order, ordered=True)

df.head()

## 7. Overview & univariate analysis

In [ ]:
print(f"Period : {df['Date_Time'].min():%d %b %Y} -> {df['Date_Time'].max():%d %b %Y}")
print(f"Sensors: {df['Sensor_Name'].nunique()}  | Rows: {len(df):,}")
print(f"Total pedestrians counted: {df['Hourly_Counts'].sum():,.0f}\n")

summary = (df.groupby("Sensor_Name")["Hourly_Counts"]
             .agg(Total="sum", Mean="mean", Median="median", Std="std", Max="max")
             .round(0).sort_values("Total", ascending=False))
summary["Share_%"] = (100 * summary["Total"] / summary["Total"].sum()).round(1)
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(df["Hourly_Counts"], bins=60, kde=True, ax=ax[0], color="#4C72B0")
ax[0].set_title("Distribution of hourly counts (all sensors)")
ax[0].set_xlabel("Pedestrians per hour")

sns.histplot(np.log1p(df["Hourly_Counts"]), bins=60, kde=True, ax=ax[1], color="#55A868")
ax[1].set_title("Log-scaled distribution  (log(1 + count))")
ax[1].set_xlabel("log(1 + count)")
plt.tight_layout(); plt.show()

print(f"Skewness: {df['Hourly_Counts'].skew():.2f}  (right-skewed -> use median / log scale for comparisons)")

## 8. Spatial analysis — *where* is footfall highest?

In [ ]:
order = summary.index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df, x="Sensor_Name", y="Hourly_Counts", order=order, estimator="mean",
            errorbar=("ci", 95), ax=ax[0], palette="viridis")
ax[0].set_title("Average hourly footfall by location (95% CI)")
ax[0].set_xlabel(""); ax[0].set_ylabel("Mean pedestrians / hour")
ax[0].tick_params(axis="x", rotation=25)

sns.boxplot(data=df, x="Sensor_Name", y="Hourly_Counts", order=order, ax=ax[1],
            palette="viridis", fliersize=1.5)
ax[1].set_title("Spread of hourly footfall by location")
ax[1].set_xlabel(""); ax[1].set_ylabel("")
ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
sns.violinplot(data=df, x="Sensor_Name", y="Hourly_Counts", hue="Day_Type", split=True,
               order=order, inner="quartile", palette="Set2", cut=0)
plt.title("Weekday vs weekend footfall distribution by location")
plt.xlabel(""); plt.ylabel("Pedestrians / hour"); plt.xticks(rotation=20)
plt.legend(title="")
plt.show()

## 9. Temporal analysis — *when* is footfall highest?
### 9.1 Hourly profile (weekday vs weekend)

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=df, x="Hour", y="Hourly_Counts", hue="Day_Type", marker="o",
             errorbar=("ci", 95), palette=["#C44E52", "#4C72B0"])
plt.xticks(range(0, 24))
plt.title("Average hourly footfall profile - all sensors")
plt.ylabel("Mean pedestrians / hour"); plt.xlabel("Hour of day")
plt.show()

In [ ]:
g = sns.relplot(data=df, x="Hour", y="Hourly_Counts", hue="Day_Type", col="Sensor_Name",
                col_wrap=3, kind="line", height=3.2, aspect=1.4, errorbar=None,
                palette=["#C44E52", "#4C72B0"], facet_kws=dict(sharey=False))
g.set_titles("{col_name}")
g.set(xticks=range(0, 24, 4))
g.fig.suptitle("Hourly profile per location - each location has its own 'personality'", y=1.03, fontweight="bold")
plt.show()

### 9.2 Day-of-week x hour heatmap

In [ ]:
heat = df.pivot_table(index="Day_Name", columns="Hour", values="Hourly_Counts", aggfunc="mean", observed=True)
plt.figure(figsize=(14, 4.8))
sns.heatmap(heat, cmap="YlOrRd", linewidths=.4, cbar_kws={"label": "Mean pedestrians / hour"})
plt.title("When is the city busiest? (mean hourly footfall, all sensors)")
plt.ylabel(""); plt.xlabel("Hour of day")
plt.show()

In [ ]:
# Location x hour heatmap (normalised so each location's pattern is comparable)
loc_hour = df.pivot_table(index="Sensor_Name", columns="Hour", values="Hourly_Counts", aggfunc="mean")
loc_hour_norm = loc_hour.div(loc_hour.max(axis=1), axis=0)

plt.figure(figsize=(14, 3.8))
sns.heatmap(loc_hour_norm, cmap="mako_r", linewidths=.4, cbar_kws={"label": "Share of that location's peak"})
plt.title("Normalised hourly pattern by location (1.0 = that location's busiest hour)")
plt.ylabel(""); plt.xlabel("Hour of day")
plt.show()

### 9.3 Weekly & seasonal patterns

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

daily = df.groupby(["Date", "Day_Name", "Sensor_Name"], observed=True)["Hourly_Counts"].sum().reset_index()
sns.barplot(data=daily, x="Day_Name", y="Hourly_Counts", hue="Sensor_Name", ax=ax[0], errorbar=None)
ax[0].set_title("Average daily footfall by weekday"); ax[0].set_xlabel(""); ax[0].set_ylabel("Pedestrians / day")
ax[0].tick_params(axis="x", rotation=30); ax[0].legend(fontsize=8, title="")

monthly = df.groupby(["Year", "Month_Name", "Sensor_Name"], observed=True)["Hourly_Counts"].sum().reset_index()
sns.lineplot(data=monthly, x="Month_Name", y="Hourly_Counts", hue="Sensor_Name", style="Year",
             markers=True, ax=ax[1])
ax[1].set_title("Monthly total footfall (2 years)"); ax[1].set_xlabel(""); ax[1].set_ylabel("Pedestrians / month")
ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
season_order = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
sns.boxplot(data=daily.assign(Season=daily["Date"].dt.month.map(season_map)),
            x="Season", y="Hourly_Counts", hue="Sensor_Name", order=season_order, fliersize=1)
plt.title("Daily footfall by season and location"); plt.ylabel("Pedestrians / day"); plt.xlabel("")
plt.legend(fontsize=8, ncol=3, title="")
plt.show()

### 9.4 Long-term trend with rolling averages

In [ ]:
daily_total = df.groupby("Date")["Hourly_Counts"].sum().rename("Total").to_frame()
daily_total["7-day avg"]  = daily_total["Total"].rolling(7,  center=True).mean()
daily_total["30-day avg"] = daily_total["Total"].rolling(30, center=True).mean()

plt.figure(figsize=(14, 5))
plt.plot(daily_total.index, daily_total["Total"], color="lightgrey", lw=.8, label="Daily total")
sns.lineplot(data=daily_total, x=daily_total.index, y="7-day avg",  color="#4C72B0", label="7-day average")
sns.lineplot(data=daily_total, x=daily_total.index, y="30-day avg", color="#C44E52", lw=2.5, label="30-day average")
plt.title("City-wide daily pedestrian footfall"); plt.xlabel(""); plt.ylabel("Pedestrians / day")
plt.show()

## 10. External drivers — weather, events and holidays
*(Available in the synthetic dataset. If you uploaded your own data, merge weather/event columns first, or skip this section.)*

In [ ]:
HAS_EXTERNAL = {"Rain", "Temperature_C", "Event"}.issubset(df.columns)
print("External drivers available:", HAS_EXTERNAL)

In [ ]:
if HAS_EXTERNAL:
    daily_loc = (df.groupby(["Date", "Sensor_Name"])
                   .agg(Total=("Hourly_Counts", "sum"), Rain=("Rain", "max"),
                        Temp=("Temperature_C", "mean"), Event=("Event", "max"),
                        Holiday=("Public_Holiday", "max"), Weekend=("Is_Weekend", "max"))
                   .reset_index())
    daily_loc["Rain_Label"] = daily_loc["Rain"].map({0: "Dry", 1: "Rain"})

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    sns.barplot(data=daily_loc, x="Sensor_Name", y="Total", hue="Rain_Label", ax=ax[0], errorbar=None,
                palette=["#F2C14E", "#4C72B0"])
    ax[0].set_title("Impact of rain on daily footfall"); ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=25)
    ax[0].set_ylabel("Pedestrians / day"); ax[0].legend(title="")

    sns.scatterplot(data=daily_loc.sample(3000, random_state=1), x="Temp", y="Total", hue="Sensor_Name",
                    alpha=.5, s=18, ax=ax[1])
    sns.regplot(data=daily_loc, x="Temp", y="Total", scatter=False, order=2, color="black", ax=ax[1],
                line_kws=dict(lw=2))
    ax[1].set_title("Temperature vs daily footfall (quadratic fit)"); ax[1].set_xlabel("Mean temperature (°C)")
    ax[1].set_ylabel(""); ax[1].legend(fontsize=7, title="")
    plt.tight_layout(); plt.show()

    # % change under rain
    pivot = daily_loc.pivot_table(index="Sensor_Name", columns="Rain_Label", values="Total", aggfunc="mean")
    pivot["Rain impact %"] = ((pivot["Rain"] - pivot["Dry"]) / pivot["Dry"] * 100).round(1)
    display(pivot.round(0).sort_values("Rain impact %"))

In [ ]:
if HAS_EXTERNAL:
    daily_loc["Event_Label"] = daily_loc["Event"].map({0: "No event", 1: "Event day"})
    plt.figure(figsize=(11, 5))
    sns.barplot(data=daily_loc[~daily_loc["Weekend"]], x="Sensor_Name", y="Total", hue="Event_Label",
                errorbar=None, palette=["#999999", "#DD8452"])
    plt.title("Event-day uplift in footfall (weekdays only, controls for weekend effect)")
    plt.xlabel(""); plt.ylabel("Pedestrians / day"); plt.xticks(rotation=20); plt.legend(title="")
    plt.show()

## 11. Statistical validation (making the EDA rigorous)

In [ ]:
# Weekday vs weekend (daily totals, per location) - Mann-Whitney U (non-parametric; counts are skewed)
rows = []
for loc, g in daily.groupby("Sensor_Name"):
    wd = g[g["Day_Name"].isin(day_order[:5])]["Hourly_Counts"]
    we = g[g["Day_Name"].isin(day_order[5:])]["Hourly_Counts"]
    u, p = stats.mannwhitneyu(wd, we)
    rows.append([loc, wd.mean(), we.mean(), (we.mean()/wd.mean()-1)*100, p])
t = pd.DataFrame(rows, columns=["Location", "Weekday mean", "Weekend mean", "Weekend vs weekday %", "p-value"]).round(3)
t["Significant (p<0.05)"] = t["p-value"] < 0.05
t

In [ ]:
if HAS_EXTERNAL:
    rows = []
    for loc, g in daily_loc.groupby("Sensor_Name"):
        g = g[~g["Weekend"]]
        dry, rain = g[g["Rain"] == 0]["Total"], g[g["Rain"] == 1]["Total"]
        u, p = stats.mannwhitneyu(dry, rain)
        rows.append([loc, dry.mean(), rain.mean(), (rain.mean()/dry.mean()-1)*100, p])
    display(pd.DataFrame(rows, columns=["Location", "Dry mean", "Rain mean", "Rain effect %", "p-value"]).round(3))

## 12. Correlation analysis

In [ ]:
# Wide table: one column per location (hourly) -> do locations move together?
wide = df.pivot_table(index="Date_Time", columns="Sensor_Name", values="Hourly_Counts")
corr = wide.corr()

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, mask=mask, ax=ax[0], square=True)
ax[0].set_title("Correlation of hourly footfall between locations")

num_cols = [c for c in ["Hourly_Counts", "Hour", "DayOfWeek", "Month", "Temperature_C", "Rain", "Event", "Public_Holiday"]
            if c in df.columns]
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax[1], square=True)
ax[1].set_title("Feature correlation matrix")
plt.tight_layout(); plt.show()

## 13. Peak-hour and congestion analysis

In [ ]:
peak = (df.groupby(["Sensor_Name", "Day_Type", "Hour"])["Hourly_Counts"].mean()
          .reset_index().sort_values("Hourly_Counts", ascending=False)
          .groupby(["Sensor_Name", "Day_Type"]).head(1).sort_values(["Sensor_Name", "Day_Type"]))
peak.columns = ["Location", "Day type", "Peak hour", "Mean pedestrians at peak"]
peak["Mean pedestrians at peak"] = peak["Mean pedestrians at peak"].round(0)
peak

In [ ]:
# Share of daily footfall by part-of-day
pod = (df.groupby(["Sensor_Name", "Part_of_Day"], observed=True)["Hourly_Counts"].sum()
         .unstack().pipe(lambda x: x.div(x.sum(axis=1), axis=0) * 100))
pod.plot(kind="barh", stacked=True, figsize=(12, 4.8), colormap="Spectral_r", width=.75)
plt.title("Share of footfall by part of day (%)"); plt.xlabel("% of total"); plt.ylabel("")
plt.legend(title="", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

## 14. Anomaly detection (unusually busy / quiet days)

In [ ]:
dl = daily.groupby(["Date", "Sensor_Name"])["Hourly_Counts"].sum().reset_index(name="Total")
dl["z"] = dl.groupby("Sensor_Name")["Total"].transform(lambda s: (s - s.mean()) / s.std())
anom = dl[dl["z"].abs() > 3].sort_values("z", ascending=False)
print(f"Anomalous location-days (|z| > 3): {len(anom)}")
display(anom.head(15))

plt.figure(figsize=(14, 4.8))
sns.scatterplot(data=dl, x="Date", y="z", hue="Sensor_Name", s=10, alpha=.5, legend=False)
sns.scatterplot(data=anom, x="Date", y="z", color="red", s=45, label="|z| > 3")
plt.axhline(3, ls="--", c="grey"); plt.axhline(-3, ls="--", c="grey")
plt.title("Daily footfall z-scores (per location)"); plt.ylabel("z-score"); plt.xlabel("")
plt.show()

## 15. One-glance dashboard

In [ ]:
fig = plt.figure(figsize=(16, 9))
gs = fig.add_gridspec(2, 3)

ax1 = fig.add_subplot(gs[0, 0]); sns.barplot(data=summary.reset_index(), y="Sensor_Name", x="Total", ax=ax1, palette="viridis")
ax1.set_title("Total footfall by location"); ax1.set_ylabel(""); ax1.set_xlabel("")

ax2 = fig.add_subplot(gs[0, 1:]); sns.lineplot(data=df, x="Hour", y="Hourly_Counts", hue="Day_Type", ax=ax2, errorbar=None, marker="o")
ax2.set_title("Hourly profile"); ax2.set_xticks(range(0, 24, 2)); ax2.set_ylabel("")

ax3 = fig.add_subplot(gs[1, :2]); sns.heatmap(heat, cmap="YlOrRd", ax=ax3, cbar=False)
ax3.set_title("Weekday x hour"); ax3.set_ylabel(""); ax3.set_xlabel("")

ax4 = fig.add_subplot(gs[1, 2]); sns.barplot(data=daily.assign(S=daily["Date"].dt.month.map(season_map)), x="S", y="Hourly_Counts",
                                             order=season_order, errorbar=None, ax=ax4, palette="crest")
ax4.set_title("Daily footfall by season"); ax4.set_xlabel(""); ax4.set_ylabel(""); ax4.tick_params(axis="x", rotation=20)

fig.suptitle("Urban Pedestrian Footfall - Executive Dashboard", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()

## 16. Auto-generated key findings

In [ ]:
busiest   = summary.index[0]
quietest  = summary.index[-1]
hour_all  = df.groupby("Hour")["Hourly_Counts"].mean()
wd_h = df[df.Day_Type == "Weekday"].groupby("Hour")["Hourly_Counts"].mean().idxmax()
we_h = df[df.Day_Type == "Weekend"].groupby("Hour")["Hourly_Counts"].mean().idxmax()
day_mean = daily.groupby("Day_Name", observed=True)["Hourly_Counts"].mean()
season_mean = daily.assign(S=daily["Date"].dt.month.map(season_map)).groupby("S")["Hourly_Counts"].mean()

print("KEY FINDINGS")
print("-" * 70)
print(f"1. Busiest location : {busiest} ({summary.loc[busiest,'Share_%']}% of all footfall); quietest: {quietest}.")
print(f"2. City-wide busiest hour: {hour_all.idxmax():02d}:00 | quietest hour: {hour_all.idxmin():02d}:00.")
print(f"3. Weekday peak hour: {wd_h:02d}:00  vs  weekend peak hour: {we_h:02d}:00.")
print(f"4. Busiest weekday: {day_mean.idxmax()} | quietest: {day_mean.idxmin()}.")
print(f"5. Busiest season: {season_mean.idxmax()} | quietest: {season_mean.idxmin()}.")
if HAS_EXTERNAL:
    worst = pivot["Rain impact %"].idxmin()
    print(f"6. Rain hits '{worst}' hardest ({pivot.loc[worst,'Rain impact %']}% on rainy days).")
    ev = daily_loc[~daily_loc.Weekend].groupby(["Sensor_Name", "Event"])["Total"].mean().unstack()
    uplift = ((ev[1] / ev[0] - 1) * 100).sort_values(ascending=False)
    print(f"7. Events boost '{uplift.index[0]}' the most (+{uplift.iloc[0]:.0f}% vs a normal weekday).")
print(f"8. Locations with strongest co-movement: "
      f"{corr.where(~np.eye(len(corr), dtype=bool)).stack().idxmax()}.")

## 17. Interpretation & planning recommendations *(CO5)*

| Insight (from the EDA) | Recommended action |
|---|---|
| Transit-linked sites show sharp **two-peak weekday** profiles (morning & evening commute) | Increase transit frequency and crowd marshals at those hours; widen footpaths/queue space near stations |
| Retail/market areas peak **midday and on weekends** | Align cleaning, waste pickup and security with the midday peak; extend weekend retail/event programming |
| Leisure sites (riverside/promenade) are **weekend- and weather-sensitive** | Provide covered walkways / shelters, promote on fair-weather weekends |
| Rain cuts footfall by different amounts at each location | Prioritise weather protection where the drop is largest |
| Events cause large **evening** surges at some sites | Pre-position crowd management & extra lighting; publish event calendars to transit operators |
| Sensor outages and spikes appear in the raw data | Add automated QA (zero-run & robust-spike checks) before data feeds dashboards |

### Limitations
- The synthetic data is illustrative; real data may include construction closures, sensor drift, COVID-style structural breaks.
- Counts measure *volume*, not *dwell time* or *origin–destination* flows.
- Observational EDA shows association, not causation — A/B pilots would be needed to confirm interventions.

### Future work
Hourly forecasting (Prophet / XGBoost / LSTM), clustering locations by temporal signature (k-means on normalised hourly profiles), and an interactive Streamlit dashboard.

## 18. Export cleaned data & figures

In [ ]:
df.to_csv("pedestrian_footfall_clean.csv", index=False)
summary.to_csv("location_summary.csv")
print("Saved: pedestrian_footfall_clean.csv, location_summary.csv")

# In Colab, uncomment to download:
# from google.colab import files
# files.download("pedestrian_footfall_clean.csv")